# NHANES Cholesterol


In [ ]:
from pathlib import Path

import pandas as pd

try:
    from experiment import prepare_nhanes, run_trials, save_results
    from plotting import plot_comparison, plot_target_qq
except ImportError:
    from dataset_ranking.experiment import prepare_nhanes, run_trials, save_results
    from dataset_ranking.plotting import plot_comparison, plot_target_qq


## Prepare data

In [ ]:
ROOT = Path.cwd()
if not (ROOT / "data").exists() and (ROOT.parent / "data").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data"
RESULT_DIR = ROOT / "results"
FIGURE_DIR = ROOT / "figures"

data = prepare_nhanes(DATA_DIR)
print("Target:", data["target_name"])
pd.DataFrame({"label": data["labels"], "dataset": data["source_names"]})


## Run experiment

In [ ]:
n_t = 30
n_k = 1000
n_test = 1000
trials = 1000
global_seed = 123
jobs = 1

data.update({
    "trials": trials,
    "seed_start": global_seed,
    "n_test": n_test,
    "n_t": n_t,
    "n_k": n_k,
})

results = run_trials(
    data,
    trials=trials,
    seed_start=global_seed,
    jobs=jobs,
    n_test=n_test,
    n_t=n_t,
    n_k=n_k,
    checkpoint_path=RESULT_DIR / "nhanes_cholesterol_checkpoint.pkl",
    checkpoint_every=4,
)
summary = save_results(data, results, RESULT_DIR)
summary


## QQ plot

In [ ]:
plot_target_qq(
    data["qq_data"],
    data["target_id"],
    data["qq_source_ids"],
    data["site_column"],
    data["response"],
    labels=data["qq_labels"],
    target_label="Target",
    title="Normal QQ Plots: NHANES Cholesterol",
    path=FIGURE_DIR / "qqplot_target_vs_sources_nhanes.png",
)


## Result plots

In [ ]:
highlights = {
    "duc": {"D4"},
    "neg_kl": {"D1", "D2", "D4"},
    "neg_score_x": {"D1", "D2"},
}
red_highlights = {name: {"D3"} for name in highlights}
mse_label_offsets = {
    "neg_score_x": {
        "D2": (-14, 8),
        "D4": (-14, 12),
    },
}

plot_comparison(
    summary,
    "Comparison of methods for data selection: NHANES Cholesterol",
    y_column="avg_rank",
    y_label="Average Rank of Test MSE (1 = Best)",
    highlight_map=highlights,
    red_highlight_map=red_highlights,
    corr_y=0.87,
    path=FIGURE_DIR / "fig_nhanes.png",
)

plot_comparison(
    summary,
    "Comparison of methods for data selection: NHANES Cholesterol",
    y_column="weighted_mse",
    y_label="Average Test MSE",
    highlight_map=highlights,
    red_highlight_map=red_highlights,
    label_offsets=mse_label_offsets,
    path=FIGURE_DIR / "fig_nhanes_mse.png",
)
